# Qrisp CUDA-Q Interface

Are you utilizing CUDA-Q in your quantum workflows? Have you wondered if this would also be possible with Qrisp as a frontend? What cane first, the chicken or the egg?

Well, as the title suggests... this tutorial answers two questions from above.

This notebook will show you how to use Qrisp's new integration with [CUDA-Q](https://nvidia.github.io/cuda-quantum/), NVIDIA's high-performance quantum programming platform. You'll see how easy it is to run Qrisp programs on CUDA-Q, use hybrid quantum/classical logic, and leverage the full power of both Python and CUDA-Q in your quantum workflows.

Long and behold, welcome to the Qrisp [CUDA-Q interface](../../reference/Jasp/CUDA-Q%20Interface.rst) tutorial!

## Hello World: Bell state

Let's start with the Hello World of quantum mechanics: the classic Bell state example. This demonstrates how to define a Qrisp kernel, **run** it with CUDA-Q, and see the results. Notice how pythonic everything ends up being.

In [1]:
import cudaq
from qrisp import *
from qrisp.jasp.cudaq_interface import cudaq_kernel, FixedShapeNDArray

@cudaq_kernel
def bell():
    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])
    return measure(qv)

# Run the kernel once (statevector simulation)
result = bell()
print("Single run result:", result)

# Run the kernel multiple times (shots)
results = cudaq.run(bell, shots_count=10)
print("10 shots:", results)

Single run result: 3
10 shots: [3, 0, 0, 0, 3, 0, 0, 3, 3, 0]


The `run` [execution mode in CUDA-Q](https://nvidia.github.io/cuda-quantum/latest/using/examples/executing_kernels.html) naturally fits Qrisp's programming model and supports versatile features like real-time hybrid control flow. However, because most current hardware backends only support sampling, Qrisp's `cudaq_kernel` decorator allows you to pass the keyword argument `execution_mode="sample"` (default is `"run"`).

Note that `sample` kernel functions must return `void`. The runtime aggregates these results across all shots, returning a `SampleResult` histogram of the raw measurement bitstrings.

In [2]:
@cudaq_kernel(execution_mode="sample")
def bell():
    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])
    measure(qv)

results = cudaq.sample(bell, shots_count=10)
print("10 shots (sample):", results)

10 shots (sample): { 00:7 11:3 }



The results contain bitstrings `00` and `11` for the Bell state $\ket{\psi}=\frac{1}{2}(\ket{00}+\ket{11})$.

## Complex Examples: Hybrid Quantum-Classical Logic

Let's shift into a higher gear now! In the following example, we'll utilize Qrisp's ability to mix classical Python (like numpy and networkx) with quantum logic, and show off hybrid control flow—where quantum measurement results influence further quantum operations.

We'll also use [Qrisp's operators module](https://www.qrisp.eu/reference/Operators/index.html) and [trotterization](https://www.qrisp.eu/reference/Operators/generated/qrisp.operators.qubit.QubitOperator.trotterization.html#qrisp.operators.qubit.QubitOperator.trotterization) for Hamiltonian simulation. This is a great way to see how expressive Qrisp kernels can be.

In [3]:
from qrisp.operators import X, Y, Z
import networkx as nx

@cudaq_kernel
def hybrid_example():
    G = nx.Graph()
    G.add_nodes_from([0, 1, 2, 3])
    G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])

    H = sum(X(i) for i in G.nodes) + sum(Z(i)*Z(j) for i, j in G.edges)
    U = H.trotterization()

    a = QuantumFloat(4)
    b = QuantumFloat(4)

    a[:] = 5
    h(b)
    U(a, t=1.0, steps=10)

    # Real-time control flow based on measurement results
    c = measure(b) < 5
    with control(c):
        a += 1

    return measure(a)

result = hybrid_example()
print("Single run result:", result)

results = cudaq.run(hybrid_example, shots_count=10)
print("10 shots:", results)

Single run result: 6.0
10 shots: [9.0, 5.0, 5.0, 14.0, 4.0, 9.0, 4.0, 6.0, 5.0, 5.0]


### Mid-Circuit Measurement with Feed-Forward

Here is another hybrid pattern that is very common in practice: measure one qubit, and conditionally apply operations to a different qubits.

This is useful for adaptive protocols, feed-forward logic, and error-correction style workflows.

In [4]:
@cudaq_kernel
def adaptive_flip_example():
    control_q = QuantumVariable(1)
    target_q = QuantumVariable(2)

    # Put control qubit into superposition, then branch on measurement.
    h(control_q[0])
    c = measure(control_q[0])

    # Apply X and Z gates to the target qubits if the control qubit is measured as 1.
    with control(c == 1):
        x(target_q[0])
        z(target_q[1])

    return measure(target_q)

print("Adaptive hybrid control (10 shots):", cudaq.run(adaptive_flip_example, shots_count=10))

Adaptive hybrid control (10 shots): [1, 0, 0, 0, 0, 1, 1, 1, 1, 0]


The result is `1` if the control is measured as `1`, and `0` if the control is measured as `0`.

### While-Loop with Accumulated Measurement

This example demonstrates [`q_while_loop`](https://www.qrisp.eu/reference/Jasp/Control%20Flow/Prefix%20Control.html#qrisp.jasp.q_while_loop): a dynamic loop whose iteration count is determined at runtime by a classical condition. The body applies a Hadamard gate to a qubit of `a` (cycling through indices modulo 4), immediately measures it, and accumulates the result into `acc`. The loop runs as long as the iteration counter is less than 4.

After the loop, the total accumulated count `acc` is used to conditionally flip the qubits of `b`: if the final value of `a` exceeds the threshold 8, an `x` gate is applied to the qubits of `b`. The kernel returns `acc + measure(b)`, combining the mid-circuit accumulation with a final measurement.

This is a good illustration of three hybrid patterns working together: a data-dependent while loop, mid-circuit measurement accumulation, and threshold-based conditional gate application.

In [5]:
from qrisp.jasp import q_while_loop

@cudaq_kernel
def main():

    a = QuantumFloat(4)
    b = QuantumFloat(4)

    def body_fun(val):
        i, acc, a = val
        h(a[i % 4])
        acc += measure(a[i % 4])
        return i + 1, acc, a

    def cond_fun(val):
        return val[0] < 4

    i, acc, a = q_while_loop(cond_fun, body_fun, (0, 0, a))

    with control(measure(a) > 8):
        x(b)

    return acc + measure(b)

print("Adaptive hybrid control (10 shots):", cudaq.run(main, shots_count=10))

Adaptive hybrid control (10 shots): [2.0, 18.0, 2.0, 18.0, 3.0, 2.0, 18.0, 17.0, 2.0, 3.0]


The final value of `acc` is between 0 and 4. The variable `a` is in a superposition state, and `b` is in state $\ket{0}$. If measuring `a` result is a value greater than 8, `b` is flipped into state $\ket{15}$. Hence, the final result is either between 0 and 4, or between 15 and 19.

## Kernel Basics: Returns, Parameters, and Arrays

Before diving into larger algorithms, it is worth covering three practical building blocks that come up in almost every non-trivial kernel:

- **Multiple return values**: a kernel can return several measurement results at once, packed into a tuple.
- **Parametrized kernels**: scalar values (`int`, `float`, `bool`) and fixed-length arrays can be passed into a kernel as classical parameters, enabling runtime-configurable circuits without recompilation.
- **Static vs dynamic arrays**: arrays created inside the kernel from `np.array` are resolved at compile time and support NumPy transformations; arrays passed in as `FixedShapeNDArray` parameters or created with `jnp.array` are dynamic and are primarily used for indexed element access inside the kernel.

The examples below illustrate each of these patterns in a minimal, executable form.

### Multiple returns

Multiple returns are supported; they are returned as a single tuple:

In [6]:
@cudaq_kernel
def main():
    a = QuantumFloat(3)
    b = QuantumFloat(2)
    a[:] = 3
    h(b)
    a += b
    return measure(a), measure(b)

print(cudaq.run(main, shots_count=5))

[(3.0, 0.0), (5.0, 2.0), (4.0, 1.0), (5.0, 2.0), (6.0, 3.0)]


The variable `a` is intialized in state $\ket{0}$, and `b` is in state $\frac{1}{4}(\ket{0}+\ket{1}+\ket{2}+\ket{3})$. Then `b` added to `a`, resulting in the entangled state $\ket{a}\ket{b}=\frac{1}{4}(\ket{3}\ket{0}+\ket{4}\ket{1}+\ket{5}\ket{2}+\ket{6}\ket{3})$. 

### Parametrized Kernels

Kernel functions decorated with `@cudaq_kernel` can accept classical parameters. The supported scalar parameter types are `int`, `float`, and `bool`. For array-valued parameters, use `FixedShapeNDArray(dtype, length)`, where `dtype` is one of those scalar types.

The example below shows a simple scalar-parametrized kernel; a single-qubit rotation by a runtime angle:

In [7]:
@cudaq_kernel
def parametrized_kernel(angle: float):
    qv = QuantumFloat(1)
    ry(angle, qv[0])
    return measure(qv)

angle = 1.57
result_static = parametrized_kernel(angle)
print("Parametrized kernel (single run):", result_static)

Parametrized kernel (single run): 1.0


#### Array Parameters: Static vs Dynamic Indexing

Array parameters are annotated with `FixedShapeNDArray(dtype, length)`. There are two important access patterns:

- **Static indexing**: the index is known at compile time (for example `angles[0]`).
- **Dynamic indexing**: the index is computed at runtime (for example from a measurement result like `angles[idx]`).

Both are supported, but dynamic indexing is more general and usually a bit more demanding for lowering/compilation. The examples below illustrate both patterns.

In [8]:
import numpy as np
import jax.numpy as jnp

# Static indexing example: access angles[0], where index is compile-time known.
@cudaq_kernel
def static_index_kernel(angles: FixedShapeNDArray(float, 3)):
    qv = QuantumFloat(1)
    ry(angles[0], qv[0])
    return measure(qv[0])

angles_static = np.array([0.0, 1.57, 3.14])
result_static = static_index_kernel(angles_static)
print("Static indexing (single run):", result_static)

shots_static = cudaq.run(static_index_kernel, angles_static, shots_count=10)
print("Static indexing (10 shots):", shots_static)

Static indexing (single run): False
Static indexing (10 shots): [False, False, False, False, False, False, False, False, False, False]


In the previous cell, `angles[0]` is fixed at compile time.

Now let's switch to **dynamic indexing**: we compute the index at runtime from a measurement result and then read `angles[idx]`.

In [9]:
# Dynamic indexing example: index is computed at runtime.
@cudaq_kernel
def dynamic_index_kernel(angles: FixedShapeNDArray(float, 3)):
    qv = QuantumVariable(1)
    idx = jnp.int32(measure(qv[0]))
    rx(angles[idx], qv[0])
    return measure(qv[0])

angles_dynamic = np.array([0.0, 1.57, 3.14])
result_dynamic = dynamic_index_kernel(angles_dynamic)
print("Dynamic indexing (single run):", result_dynamic)

shots_dynamic = cudaq.run(dynamic_index_kernel, angles_dynamic, shots_count=10)
print("Dynamic indexing (10 shots):", shots_dynamic)

Dynamic indexing (single run): False
Dynamic indexing (10 shots): [False, False, False, False, False, False, False, False, False, False]


### Static vs Dynamic Arrays in CUDA-Q Kernels

This topic is important, because array behavior changes depending on where the array comes from.

- **Static array**: created inside the kernel body (for example via `np.array([...])`).
- **Dynamic array parameter**: passed into the kernel and annotated as `FixedShapeNDArray(dtype, n)` or created inside the kernel (i.e., `jnp.array[...])`).

Practical rule of thumb:

- Static arrays defined inside the kernel support standard array transformations (for example `np.sin`).
- Dynamic array parameters are primarily meant for indexed access (`angles[i]`) and passing values into quantum operations. Transformations on dynamic arrays inside a kernel (for example `jnp.sin`) are not supported at this point.

The three examples below demonstrate these patterns in executable form.

In [10]:
# Example 1: static array created inside the kernel
@cudaq_kernel
def static_array_kernel():
    qv = QuantumFloat(5)

    angles = np.array([1.57, 0.78, 0.39, 0.25, 0.12])
    angles = np.sin(angles)

    # Static array with static loop.
    for i in range(5):
        ry(angles[i], qv[i])

    return measure(qv)

print("Static array in-kernel (10 shots):", cudaq.run(static_array_kernel, shots_count=10))

Static array in-kernel (10 shots): [1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 8.0, 0.0]


In [11]:
# Example 2: dynamic array passed as kernel parameter
@cudaq_kernel
def dynamic_array_kernel(angles: FixedShapeNDArray(float, 5)):
    qv = QuantumFloat(5)

    # Dynamic parameter array accessed element-wise.
    for i in jrange(5):
        ry(angles[i], qv[i])

    return measure(qv)

angles = np.array([1.57, 0.78, 0.39, 0.25, 0.12])
print("Dynamic array parameter (10 shots):", cudaq.run(dynamic_array_kernel, angles, shots_count=10))

Dynamic array parameter (10 shots): [0.0, 0.0, 1.0, 5.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0]


In [12]:
# Example 3: unsupported dynamic array operations inside the kernel (should fail to compile)
try:
    @cudaq_kernel
    def dynamic_array_bad_kernel():
        qv = QuantumFloat(5)

        # Operations like jnp.sin on dynamic arrays are not currently supported inside the kernel.
        angles = jnp.array([1.57, 0.78, 0.39, 0.25, 0.12])
        angles = jnp.sin(angles)

        # Static array with static loop.
        for i in range(5):
            ry(angles[i], qv[i])

        return measure(qv)
except RuntimeError as e:
    print("Error defining kernel:", e)

Error defining kernel: Failed to compile Qrisp function 'dynamic_array_bad_kernel' to MLIR: Cannot lower this program to CUDA-Q because it still contains a ranked-tensor linalg.generic operation.

What this usually means:
- You performed arithmetic on traced jax.numpy arrays (for example array expressions inside the traced function), which introduced ranked tensor linalg ops.

How to fix it:
- Prefer scalar values inside the traced kernel body.
- Move array arithmetic outside tracing and pass only the needed elements/scalars into the kernel.
- If you pass arrays as kernel parameters, stick to simple indexed access patterns.

Location: loc(unknown)
Offending operation: GenericOp(
	%0 = linalg.generic {indexing_maps = [affine_map<(d0) -> (d0)>, affine_map<(d0) -> (d0)>], iterator_types = ["parallel"]} ins(%1 : tensor<5xf64>) outs(%2 : tensor<5xf64>) {
	^bb0(%arg5: f64, %arg6: f64):
	  %3 = math.sin %arg5 : f64
	  linalg.yield %3 : f64
	} -> tensor<5xf64>
)


## Example: Amplitude Amplification

Amplitude amplification is the generalization of Grover's algorithm. It boosts the probability of measuring a target state by repeatedly applying a state-preparation step and an oracle phase flip.

The kernel here takes an `int` parameter `iterations`, the number of amplification iterations, which is a good example of an integer parametrized kernel.

The setup is:

- **State preparation**: a small `ry(π/8)` rotation initialises the qubit in a state with low initial overlap with `|True⟩`.
- **Oracle**: a `z` gate flips the phase of the `|True⟩` component.
- **Amplitude amplification**: each call to `amplitude_amplification(..., iter=iterations)` applies `iterations` Grover-style reflection pairs.

The example runs the kernel for `iter = 1, 2, 3` and prints the fraction of `True` outcomes over 100 shots. Each additional iteration boosts the success probability, illustrating how the number of iterations directly controls the amplification strength.

In [13]:
from qrisp.alg_primitives import amplitude_amplification

@cudaq_kernel
def amplitude_amplification_example(iterations: int):
    def state_function(qb):
        ry(np.pi / 8, qb)

    def oracle_function(qb):
        z(qb)

    qb = QuantumBool()
    state_function(qb)
    amplitude_amplification([qb], state_function, oracle_function, iter=iterations)
    return measure(qb[0])

shots = 100
for it in [0, 1, 2, 3]:
    result = cudaq.run(amplitude_amplification_example, it, shots_count=shots)
    ones_fraction = sum(result) / len(result)
    print(f"iterations={it} -> fraction of 1 outcomes: {ones_fraction:.3f}")
    print(f"iterations={it} -> raw shots: {result}\n")

iterations=0 -> fraction of 1 outcomes: 0.070
iterations=0 -> raw shots: [False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, True, False, True, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False]

iterations=1 -> fraction of 1 outcomes: 0.300
iterations=1 -> raw shots: [False, False, False, False, True, True, False, False, False, True, False, False, False, False, False, False, False, False, False, False, True, False, False, F

We see that the probability of measuring the variable as `True` grows with increasing the number of iterations. 

## Example: Custom MaxCut QAOA with CUDA-Q + SciPy

This example puts everything together into a full hybrid variational loop.

- The kernel takes a parameter array `params = [gamma_0, gamma_1, beta_0, beta_1]`.
- The **cost operator** applies MaxCut phase separators along the graph edges.
- The **mixer** applies `rx` rotations to every qubit.
- A classical SciPy loop repeatedly calls `cudaq.run(...)` and optimizes the expected cut value computed from the measured bitstrings.

In [14]:
from scipy.optimize import minimize
import networkx as nx
import numpy as np

# 1. Define the MaxCut Instance
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (0, 2), (1, 3), (1,4), (2,5), (3,6), (4,5), (5,6), (6,4)]
N = 7
num_layers = 2

def apply_cost_operator(qv, gamma):
    for u, v in EDGES:
        cx(qv[u], qv[v])
        rz(2 * gamma, qv[v])
        cx(qv[u], qv[v])

def apply_mixer(qv, beta):
    for i in range(N):
        rx(2 * beta, qv[i])

# 2. Define the Qrisp CUDA-Q Kernel
@cudaq_kernel
def variational_maxcut_kernel(params: FixedShapeNDArray(float, 2 * num_layers)):
    qv = QuantumVariable(N)
    
    # Start in the uniform superposition
    h(qv)

    num_layers = len(params) // 2
    
    # Apply multiple QAOA layers
    for i in jrange(num_layers):
        apply_cost_operator(qv, params[i])     # Gammas
        apply_mixer(qv, params[i + num_layers])         # Betas
        
    return measure(qv)

# 3. Simplified Classical Post-Processing
def calculate_cut_value(bitstring: int) -> int:
    """Calculates the cut size for a given measured integer bitstring."""
    cut = 0
    for u, v in EDGES:
        bit_u = (bitstring >> u) & 1
        bit_v = (bitstring >> v) & 1
        if bit_u != bit_v:
            cut += 1
    return cut

def objective_function(params: np.ndarray) -> float:
    """Runs the quantum kernel and evaluates the expected (average) cut."""
    # Sample the circuit
    samples = cudaq.run(variational_maxcut_kernel, params, shots_count=100)
    
    # Calculate the average cut value across all shots
    avg_cut = np.mean([calculate_cut_value(int(s)) for s in samples])
    
    # Scipy minimizes, so we return the negative expected cut
    return -avg_cut

# 4. Hybrid Optimization Loop
np.random.seed(42)
initial_params = np.random.uniform(0.0, np.pi, size=4)

print(f"Starting optimization... Initial expected cut: {-objective_function(initial_params):.3f}")

opt_result = minimize(
    objective_function,
    initial_params,
    method="COBYLA",
    options={"maxiter": 40}
)

# 5. Evaluate the Best Result 
best_params = opt_result.x
final_samples = cudaq.run(variational_maxcut_kernel, best_params, shots_count=100)

# Find the best bitstring from our final heavily-sampled distribution
best_sample = max([int(s) for s in final_samples], key=calculate_cut_value)

print("\nOptimization finished! ✨")
print(f"Final expected cut: {-opt_result.fun:.3f}")
print(f"Best bitstring found: {bin(best_sample)[2:].zfill(N)}")
print(f"Max cut value achieved: {calculate_cut_value(best_sample)}")
print(f"Optimized parameters: {np.round(best_params, 3)}")

Starting optimization... Initial expected cut: 5.830

Optimization finished! ✨
Final expected cut: 6.380
Best bitstring found: 1010101
Max cut value achieved: 9
Optimized parameters: [0.821 3.049 2.348 2.981]


We have successfully run a hybrid quantum-calssical optimization loop solving our MacCut problem. The solution is encoded by the best bitsring found. 

## Recap

So far we've covered a lot of ground and touched on a plethora of topics. Let's gather them into a list:

- **Bell state & basic kernels**: defining and running `@cudaq_kernel` functions, single-shot and multi-shot execution with `cudaq.run`.
- **Hybrid quantum-classical logic**: mixing classical Python (NumPy, NetworkX), real-time control flow (`control`, `q_cond`), and mid-circuit measurements.
- **Multiple return values**: returning tuples of measurement results.
- **Parameterized kernels**: scalar (`int`, `float`, `bool`) and array (`FixedShapeNDArray`) parameters, with both static and dynamic indexing.
- **Amplitude amplification**: a non-trivial algorithm as an integer-parametrized kernel.
- **QAOA**: a full hybrid variational loop with SciPy optimization over a 7-node MaxCut instance.

All of this runs natively on CUDA-Q. But the real question is: can we cover even more ground?

[Block encodings](https://www.qrisp.eu/general/tutorial/BE_tutorial/BE_vol1.html) are one of the most powerful and mathematically sophisticated tools in Qrisp. They represent linear operators as the top-left block of a unitary, enabling polynomial transformations of spectra, matrix inversion, Hamiltonian simulation, and much more. If the CUDA-Q interface can handle block encodings, it can handle essentially anything Qrisp throws at it.

So, can this interface run block encodings? Let's find out!

## Final Example: Block Encodings

A [block encoding](https://arxiv.org/abs/1806.01838) of a matrix $A$ is a unitary $U$ such that

$$ A = (\langle 0 \otimes I) | U | (0 \rangle \otimes I) $$

i.e., $A$ appears as the top-left block of $U$ when the ancilla variables are projected onto $|0\rangle$. This construction is the foundation of quantum singular value transformation (QSVT) and enables a broad class of quantum algorithms: Hamiltonian simulation, quantum linear systems solvers, spectral filtering, and more.

Qrisp's [BlockEncoding](../../reference/Block%20Encodings/BlockEncoding.rst) class lets you construct block encodings directly from operator expressions and compose them with `+`, `-`, `@`, `.poly()`, `.inv()`, `.sim()`, and so on, all **outside** the kernel, since those compositions involve classical arithmetic on traced JAX arrays, which is not supported inside the tracing context.

Inside the kernel, you call `BE.apply(operand)` to apply the block encoding and obtain the ancilla variables. Because the encoding is only exact up to post-selection on the ancillas being in $|0\rangle$, the kernel returns **both** the operand measurement and the ancilla measurement. The classical post-processing then filters for the `ancilla == 0` outcomes.

In [15]:
from collections import Counter
from qrisp.block_encodings import BlockEncoding
from qrisp.operators import X, Y, Z


def post_selection(res_dict):
    # Post-selection on ancillas being in |0> state
    filtered_dict = {k[0]: p for k, p in res_dict.items() \
                    if all(x == 0 for x in k[1:])}
    success_prob = sum(filtered_dict.values())
    filtered_dict = {k: p / success_prob for k, p in filtered_dict.items()}
    return filtered_dict


# Define an Ising-Hamiltonian and its block encoding outside of the kernel
H = Z(0)*Z(1) + Z(1)*Z(2) + Z(2)*Z(3) + X(0) + X(1) + X(2) + X(3)
BE = BlockEncoding.from_operator(H)

@cudaq_kernel
def main():

    operand = QuantumVariable(4)
    ancs = BE.apply(operand)
    return measure(operand), measure(ancs[0])

results = cudaq.run(main, shots_count=100)
counts = Counter(results)
res_dict = {outcome: count / len(results) for outcome, count in counts.items()}
filtered_dict = post_selection(res_dict)
print("Post-selected distribution:", filtered_dict)

Post-selected distribution: {4: 0.08333333333333334, 0: 0.6666666666666667, 8: 0.08333333333333334, 1: 0.16666666666666669}


This example shows how to apply the **non-unitary** Ising Hamiltonian $H$ to a variable in initial state $\ket{\psi_0}=\ket{0}$. The block encoding is applied sucessfully, if the ancilla variable is measured in state $\ket{0}$. The post-selected distribution shows the measurement distribution of the state $\ket{\psi}=H\ket{\psi_0} / \lVert H\ket{\psi_0}\rVert$.

You can combine all these features to build powerful, flexible quantum-classical workflows!

## Epilogue



---

## Tips, Gotchas, and Debugging

- **Kernel parameters:** Use `FixedShapeNDArray` for numpy arrays, and regular Python types for scalars.
- **Hybrid control:** You can use `measure()`, `control()`, `q_cond()`, `q_while_loop()`, and `q_fori_loop()` to build hybrid quantum-classical logic.
- **Debugging:** If you get errors, check your kernel signature and make sure all types are supported by CUDA-Q.
- **Limitations:** Some advanced Qrisp features may not yet be supported inside kernels. For that reason, keep kernels simple and explicit for best results.

If you run into trouble, open an issue or ask other Qrisp devs for help and/or insights on the [Eclipse Qrisp Discord server](https://discord.gg/v5np7DeBaq).

---